In [287]:
import numpy as np
import pandas as pd
import torch
from torch import nn 

In [288]:
def train_test_split(X, y, test_size:float = None, train_size:float = None):
	if X.shape[0] != y.shape[0]:
		raise Exception("Size mismatch of X and y")
	else:
		total_rows=X.shape[0]
	if test_size == None and train_size == None:
		test_size=0.25
		train_size = 1 - test_size
	elif test_size == None:
		test_size = 1 - train_size
	elif train_size == None:
		train_size = 1 - test_size

	mark = int(total_rows*train_size)
	return X[:mark], y[:mark], X[mark:], y[mark:]


In [289]:
# from sklearn.model_selection import train_test_split
df = pd.read_csv('data.csv')
df.head()
X = df.drop(columns=['price']).to_numpy()
y = df['price'].to_numpy()

In [290]:
X_train, y_train, X_test, y_test = train_test_split(X, y, 0.2)
# Scaling
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)
X_train

array([[ 750,    1],
       [ 900,    2],
       [1200,    2],
       [1500,    3],
       [1800,    3],
       [2000,    4],
       [2200,    4],
       [2500,    4]])

In [312]:
class Perceptron():
	def __init__(self, features_size):
		self.weights = torch.tensor(np.random.random(features_size), requires_grad=True, dtype=torch.float32)
		self.bias = torch.tensor(np.random.random(), requires_grad=True, dtype=torch.float32)

	def forward(self, features):
		features = torch.tensor(features, dtype=torch.float32)
		self.model_pred = torch.empty(features.shape[0], dtype=torch.float32)
		for i in range(features.shape[0]):
			self.model_pred[i] = (torch.dot(self.weights, features[i]) + self.bias)
		return self.model_pred
			
	def rmse(self, y_true):
		y_true = torch.from_numpy(y_true)
		if y_true.shape != self.model_pred.shape:
			raise Exception("Shape mismatch(y_true and y_pred are not same shape)")
		num = y_true.numel()
		# self.loss=torch.empty(1)
		self.loss = ((1/num) * sum(y_true - self.model_pred)**2)**0.5

		# loss = ((sum((y_true - y_pred)**2))**0.5)/num
		# self.loss = torch.tensor(((sum(y_true - y_pred)**2)**0.5)/num, requires_grad=True)
		return self.loss
	
	def backpropagation(self, lr=0.01):
		self.loss.backward()
		with torch.no_grad():
			for i in range(len(self.weights)):
				self.weights[i] = self.weights[i] - (lr * self.weights.grad[i])
			self.bias = self.bias - (lr * self.bias.grad)

	def zero_grads(self):
		torch.nn.Module.zero_grad(self)


In [313]:
model = Perceptron(X_train.shape[1])
epochs = 10
# for i in range(epochs):
y_pred = (model.forward(X_train))

print("Before backprop weights:", model.weights)
# Calculate loss

loss = model.rmse(y_train)
print("loss:", loss)
# update weights 
model.backpropagation(lr=0.01)

print(model.bias.grad)



Before backprop weights: tensor([0.7464, 0.1841], requires_grad=True)
loss: tensor(373141.4062, grad_fn=<PowBackward0>)
None


In [314]:
print(model.bias.grad)

None


In [294]:
y_t = np.array([1])
y_p = np.array([0.5])
((sum((y_t - y_p)**2))**0.5)/len(y_t)


np.float64(0.5)

In [295]:
t = torch.empty(X_train.shape[0])
for i in range(5):
	t[i] += i
t

tensor([ 4.4575e-11,  1.0000e+00,  1.6464e+00,  2.6464e+00,  3.6464e+00,
        -3.5355e-01, -3.5355e-01,  0.0000e+00])